# Chapter 2: Probability Theory Fundamentals

**Book**: Michael Brenndoerfer – Quantitative Finance  

This notebook covers key probability concepts used in quantitative finance:
- Discrete probability distributions
- Expected value
- Variance and standard deviation
- Kurtosis (fat tails)
- Bayes’ theorem
- Value at Risk (VaR)

### 1. Credit Rating Probability Model

In credit risk, a bond’s credit rating can change over time.  
Here we model three possible states at year-end:

- **Upgraded**
- **Unchanged**
- **Downgraded**

We assign probabilities to each state and verify that they sum to 1 (a basic requirement of any probability distribution).

In [ ]:
import numpy as np

states = ["Upgraded", "Unchanged", "Downgraded"]
probabilities = np.array([0.15, 0.70, 0.15])

assert np.isclose(probabilities.sum(), 1.0)
print("P(Upgraded): ", probabilities[0])
print("P(Unchanged): ", probabilities[1])
print("P(Degraded): ", probabilities[2])
print("Sum of the probabilities ", probabilities[0]+probabilities[1]+probabilities[2])

### 2. Expected Value of Returns

The **expected value** (or expected return) is the probability-weighted average of all possible outcomes.

Formula:

\[
E[R] = \sum (x_i \times p_i)
\]

This is one of the most fundamental concepts in finance — it tells us the average return we can expect from an investment.

In [1]:
import numpy as np

# Possible percentage returns (+20% or -10%)
outcomes = np.array([0.20, -0.10])

# Assigned probabilities for each state
probabilities = np.array([0.60, 0.40])

# Calculate expected return: E[R] = Sum(x * p)
expected_return = np.sum(outcomes * probabilities)

print(f"Expected Return: {expected_return:.2f}") 
# Output: Expected Return: 0.08


Expected Return: 0.08


### 3. Variance and Standard Deviation

While expected return tells us the *average* outcome, it does not tell us about *risk*.

- **Variance** measures how spread out the possible returns are.
- **Standard Deviation (σ)** is the square root of variance and is the most common measure of volatility/risk.

Higher standard deviation = higher risk.

In [3]:
import numpy as np

outcomes = np.array([0.20, -0.10])  # +20% and -10%
probabilities = np.array([0.60, 0.40])  # 60% and 40%

expected_return = np.sum(outcomes * probabilities)

expected_return_squared = np.sum((outcomes**2) * probabilities)

variance = expected_return_squared - (expected_return**2)

std_dev = np.sqrt(variance)

print(f"Expected Return E[R]     : {expected_return:.2%}")
print(f"Expected Return Sq E[R²] : {expected_return_squared:.4f}")
print(f"Variance Var(R)          : {variance:.4f}")
print(f"Standard Deviation (σ)   : {std_dev:.2%}")


Expected Return E[R]     : 8.00%
Expected Return Sq E[R²] : 0.0280
Variance Var(R)          : 0.0216
Standard Deviation (σ)   : 14.70%


### 4. Kurtosis and Fat Tails

**Kurtosis** measures the “tailedness” of a distribution.

- Normal distribution has excess kurtosis ≈ 0
- Distributions with **high kurtosis** have fatter tails (higher chance of extreme events)

In finance, returns often show fat tails — meaning extreme losses (or gains) happen more frequently than a normal distribution would predict. This is why risk models based only on normal distribution can underestimate risk.

In [2]:
import numpy as np
from scipy.stats import kurtosis, t

np.random.seed(42)
n_samples = 100000

normal_samples = np.random.randn(n_samples)
t_samples = t.rvs(df=4, size=n_samples)

normal_kurtosis = kurtosis(normal_samples)
t_kurtosis = kurtosis(t_samples)

normal_tail_prob = np.mean(np.abs(normal_samples) > 4) * 100
t_tail_prob = np.mean(np.abs(t_samples) > 4) * 100

print(f"Normal distribution excess kurtosis: {normal_kurtosis:.3f}")
print(f"Student's t (df=4) excess kurtosis: {t_kurtosis:.3f}\n")
print("Probability of |X| > 4:")
print(f"Normal distribution: {normal_tail_prob:.4f}%")
print(f"Student's t (df=4): {t_tail_prob:.4f}%")


Normal distribution excess kurtosis: -0.008
Student's t (df=4) excess kurtosis: 17.487

Probability of |X| > 4:
Normal distribution: 0.0050%
Student's t (df=4): 1.5800%


### 5. Bayes’ Theorem – Updating Default Probability

Bayes’ theorem allows us to update our beliefs when new information arrives.

In this example:
- Prior probability of default = 3%
- We observe “deteriorating margins”
- We update the probability of default given this new information

This is widely used in credit risk modeling and machine learning.

In [ ]:

p_default = 0.03       # P(D): Prior probability of default (3%)
p_no_default = 0.97    # P(D'): Prior probability of non-default (97%)

p_margin_given_default = 0.40      # Likelihood of deteriorating margins given default
p_margin_given_no_default = 0.10   # Likelihood of deteriorating margins given no default

p_margin = (
    p_margin_given_default * p_default
    + p_margin_given_no_default * p_no_default
)


p_default_given_margin = (p_margin_given_default * p_default) / p_margin

# Calculating how many times the risk increased relative to the prior
increase_factor = p_default_given_margin / p_default
print(f"P(Deteriorating Margins) = {p_margin_given_default} × {p_default} + {p_margin_given_no_default} × {p_no_default}")
print(f"P(Deteriorating Margins) = {p_margin:.4f}\n")

print(f"P(Default | Deteriorating Margins) = ({p_margin_given_default} × {p_default}) / {p_margin:.4f}")
print(f"P(Default | Deteriorating Margins) = {p_default_given_margin:.4f}\n")

print(f"Updated default probability: {p_default_given_margin * 100:.2f}%")
print(f"Prior default probability:   {p_default * 100:.2f}%")
print(f"Increase factor:             {increase_factor:.1f}x")

P(Deteriorating Margins) = 0.4 × 0.03 + 0.1 × 0.97
P(Deteriorating Margins) = 0.1090

P(Default | Deteriorating Margins) = (0.4 × 0.03) / 0.1090
P(Default | Deteriorating Margins) = 0.1101

Updated default probability: 11.01%
Prior default probability:   3.00%
Increase factor:             3.7x


### 6. Simulating Stock Returns

We simulate daily returns using a normal distribution based on:
- Expected annual return
- Annual volatility

This is a common technique used in Monte Carlo simulations for pricing derivatives and risk management.

In [1]:
import numpy as np

np.random.seed(42)

annual_return = 0.10  # 10% expected annual return
annual_volatility = 0.20  # 20% annual volatility

#(252 trading days)
trading_days = 252
daily_return = annual_return / trading_days
daily_volatility = annual_volatility / np.sqrt(trading_days)

n_years = 5
n_days = trading_days * n_years

print("Simulation Parameters:")
print(f"  Expected annual return: {annual_return * 100:.1f}%")
print(f"  Annual volatility: {annual_volatility * 100:.1f}%")
print(f"  Daily expected return: {daily_return * 100:.4f}%")
print(f"  Daily volatility: {daily_volatility * 100:.4f}%\n")

print(f"Simulated {n_days} trading days ({n_years} years)")

Simulation Parameters:
  Expected annual return: 10.0%
  Annual volatility: 20.0%
  Daily expected return: 0.0397%
  Daily volatility: 1.2599%

Simulated 1260 trading days (5 years)


### 7. Value at Risk (VaR)

**Value at Risk (VaR)** estimates the maximum potential loss over a given time period at a certain confidence level.

We calculate two types:
- **Historical VaR**: Based on the actual simulated returns
- **Parametric VaR**: Assumes returns follow a normal distribution

VaR is one of the most widely used risk measures in the financial industry.

In [ ]:
import numpy as np
from scipy import stats


sample_mean = np.mean(returns)
sample_std = np.std(returns, ddof=1)

var_95_hist = -np.percentile(returns, 5)
var_99_hist = -np.percentile(returns, 1)

var_95_param = -(sample_mean + stats.norm.ppf(0.05) * sample_std)
var_99_param = -(sample_mean + stats.norm.ppf(0.01) * sample_std)

print(" Value at Risk (Daily):")
print(f"95% VaR:")
print(f"  Historical:          {var_95_hist * 100:.4f}%")
print(f"  Parametric (Normal): {var_95_param * 100:.4f}%")
print()
print(f"99% VaR:")
print(f"  Historical:          {var_99_hist * 100:.4f}%")
print(f"  Parametric (Normal): {var_99_param * 100:.4f}%")

# Summary Interpretations
print(f"\nInterpretation (95% Historical):")
print(f"  With 95% confidence, daily losses will not exceed {var_95_hist * 100:.2f}%.")
print(f"  (There is a 5% probability that daily losses will be greater than {var_95_hist * 100:.2f}%).")

 Value at Risk (Daily)
95% VaR:
  Historical:          1.9143%
  Parametric (Normal): 1.9629%

99% VaR:
  Historical:          2.6514%
  Parametric (Normal): 2.8125%

Interpretation (95% Historical):
  With 95% confidence, daily losses will not exceed 1.91%.
  (There is a 5% probability that daily losses will be greater than 1.91%).
